# TPU v5e-1 feasibility probe — SkySim PPO

This is an isolated hardware compatibility benchmark, not the 40-million-step training run. **Current Colab TPU runtimes use TorchTPU**, so this notebook checks `torch_tpu` / `device='tpu'` rather than the older PyTorch/XLA interface.

Important limits:
- The current project trains PPO with Stable-Baselines3 and calls the native C++ DroneCore simulator through `ctypes`.
- Selecting a TPU does **not** move that C++ physics simulation to TPU, and Stable-Baselines3 does not provide a drop-in TPU device setting.
- This benchmark measures only neural-network compute, not end-to-end PPO speed.
- The production target remains one shared policy, 40 parallel environments, `N_STEPS=1000`, `BATCH_SIZE=4000`, and 40,000,000 environment transitions. This probe does not modify those settings or start that run.

## 1. Verify the selected Colab runtime
In Colab, choose **Runtime → Change runtime type → TPU v5e-1**, save, then use **Runtime → Restart session** if you changed the accelerator. Run the notebook from the top. Current Colab TPU images use TorchTPU rather than the older `torch_xla` setup.

In [ ]:
import os, platform, sys, importlib.util
print('Python:', sys.version)
print('Platform:', platform.platform())
print('COLAB_TPU_ADDR set:', bool(os.environ.get('COLAB_TPU_ADDR')))
print('TPU_VISIBLE_DEVICES:', os.environ.get('TPU_VISIBLE_DEVICES'))
print('torch_tpu module spec:', importlib.util.find_spec('torch_tpu'))

import torch
print('PyTorch:', torch.__version__)
print('CUDA visible:', torch.cuda.is_available())

TPU_AVAILABLE = False
TPU_DEVICE = None
try:
    import torch_tpu  # registers the TorchTPU backend in current Colab TPU runtimes
    TPU_DEVICE = torch.device('tpu')
    probe = torch.ones((2, 2), device=TPU_DEVICE)
    result = (probe @ probe).cpu()  # force a result back to CPU to surface device errors
    print('TorchTPU import: OK')
    print('TPU device:', TPU_DEVICE)
    print('TPU smoke-test result:', result)
    TPU_AVAILABLE = True
except Exception as exc:
    print('TorchTPU initialization failed:', type(exc).__name__, str(exc))
    print('\nDiagnostics:')
    print('- If torch_tpu module spec is None, this runtime image may not have TorchTPU installed.')
    print('- If the module exists but device initialization fails, check that the session really is using TPU v5e-1 and restart the runtime.')
    print('- Do not install an arbitrary torch/torch_xla version over the Colab-provided PyTorch stack; send this full cell output for diagnosis.')

## 2. Benchmark a representative small MLP on CPU vs TPU
This is a synthetic neural-network workload, not PPO. TorchTPU is benchmarked with a warm-up and a final device-to-CPU copy to ensure measured work has completed.

In [ ]:
import time
from torch import nn

if not TPU_AVAILABLE:
    raise RuntimeError('TorchTPU did not initialize. Share the full output from section 1; this is a runtime/backend issue, not a benchmark result.')

torch.manual_seed(42)
BATCH = 1024
INPUT_DIM = 64
HIDDEN_DIM = 256
OUTPUT_DIM = 4
WARMUP_STEPS = 20
MEASURE_STEPS = 100

def benchmark_mlp(device):
    model = nn.Sequential(
        nn.Linear(INPUT_DIM, HIDDEN_DIM), nn.Tanh(),
        nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.Tanh(),
        nn.Linear(HIDDEN_DIM, OUTPUT_DIM)
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    x = torch.randn(BATCH, INPUT_DIM, device=device)
    y = torch.randn(BATCH, OUTPUT_DIM, device=device)
    last_loss = None

    def one_step():
        nonlocal last_loss
        optimizer.zero_grad(set_to_none=True)
        prediction = model(x)
        last_loss = ((prediction - y) ** 2).mean()
        last_loss.backward()
        optimizer.step()

    for _ in range(WARMUP_STEPS):
        one_step()
    _ = last_loss.detach().cpu().item()  # synchronize before timing

    start = time.perf_counter()
    for _ in range(MEASURE_STEPS):
        one_step()
    final_loss = last_loss.detach().cpu().item()  # synchronize measured work
    elapsed = time.perf_counter() - start
    return {'device': str(device), 'seconds': round(elapsed, 3),
            'steps_per_second': round(MEASURE_STEPS / elapsed, 2),
            'samples_per_second': round(MEASURE_STEPS * BATCH / elapsed, 1),
            'final_loss': round(final_loss, 6)}

cpu_result = benchmark_mlp(torch.device('cpu'))
tpu_result = benchmark_mlp(TPU_DEVICE)
print('CPU:', cpu_result)
print('TPU:', tpu_result)
print('TPU / CPU step-throughput ratio:', round(tpu_result['steps_per_second'] / cpu_result['steps_per_second'], 2))

## 3. Interpretation and next step

Share the full section 1 output and the metrics from section 2.

- If TorchTPU initialization fails, do not attempt the long training run until the runtime issue is resolved.
- A faster synthetic MLP only shows that the accelerator can help neural-network math. It does **not** prove the current Stable-Baselines3 PPO pipeline can use TPU or that end-to-end training will be faster.
- Moving actual training to TPU requires a compatible PPO training path and validation of equivalent observations, actions, rewards, and update behavior. The existing C++ physics remains CPU-bound unless separately ported/vectorized.
- Do not change the production training constants as part of this experiment.